# Notebook 04 — Inferência Local vs Remota

**Objetivo:** Comparar execucao local (GPT4All Phi-3-mini) vs remota (OpenAI GPT-4o-mini)
em 5 dimensoes: qualidade, latencia, custo, privacidade e controle.

**Rubricas cobertas:** Rubrica 4 — todos os 5 itens.

## 7.1 Setup — Ambos os Backends

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")

In [ ]:
import sys
sys.path.insert(0, '.')

import os
import time
import json
import re
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sklearn.metrics import accuracy_score, f1_score, classification_report

from scripts.config import ANOTACOES_DIR, CLASSES

load_dotenv()

# Verificar API key
openai_key = os.getenv('OPENAI_API_KEY', '')
print(f'OpenAI key configurada: {"sim" if openai_key else "nao — células que usam OpenAI serão puladas"}')

In [ ]:
import os
import time
import json
from dotenv import load_dotenv
load_dotenv()

class LLMProvider:
    """Interface unificada para DeepSeek (remoto), OpenAI (remoto) e GPT4All (local)."""
    
    def __init__(self, backend: str = 'deepseek'):
        self.backend = backend
        
        if backend == 'deepseek':
            self.api_key = os.getenv('DEEPSEEK_API_KEY')
            if not self.api_key:
                print('AVISO: DEEPSEEK_API_KEY nao configurada no .env.')
        elif backend == 'openai':
            import openai
            openai.api_key = os.getenv('OPENAI_API_KEY')
            if not openai.api_key:
                print('AVISO: OPENAI_API_KEY nao configurada.')
        elif backend == 'gpt4all':
            try:
                from gpt4all import GPT4All
                self.model = GPT4All('Phi-3-mini-4k-instruct.Q4_K_M.gguf',
                                    allow_download=True)
                print('GPT4All Phi-3-mini carregado (local).')
            except Exception as e:
                print(f'Erro ao carregar GPT4All: {e}')
                self.model = None

    def generate(self, prompt: str, max_tokens: int = 200,
                 temperature: float = 0.0) -> str:
        """
        Gera resposta para o prompt.
        Tenta ate 3 vezes em caso de erro.
        """
        import urllib.request
        import urllib.error
        
        if self.backend == 'deepseek':
            if not self.api_key:
                return ''
            model = os.getenv('DEEPSEEK_MODEL', 'deepseek-chat')
            url = 'https://api.deepseek.com/chat/completions'
            payload = {
                'model': model,
                'messages': [{'role': 'user', 'content': prompt}],
                'max_tokens': max_tokens,
                'temperature': temperature,
            }
            headers = {
                'Authorization': f'Bearer {self.api_key}',
                'Content-Type': 'application/json',
            }
            for attempt in range(3):
                try:
                    req = urllib.request.Request(
                        url,
                        data=json.dumps(payload).encode('utf-8'),
                        headers=headers,
                        method='POST',
                    )
                    with urllib.request.urlopen(req, timeout=60) as resp:
                        data = json.loads(resp.read().decode('utf-8'))
                    return data['choices'][0]['message']['content']
                except urllib.error.HTTPError as e:
                    print(f'Tentativa {attempt+1} falhou (HTTP {e.code}): {e.read().decode()}')
                    time.sleep(2 ** attempt)
                except Exception as e:
                    print(f'Tentativa {attempt+1} falhou: {e}')
                    time.sleep(2 ** attempt)
            return ''
        elif self.backend == 'openai':
            if not os.getenv('OPENAI_API_KEY'):
                return ''
            import openai
            for attempt in range(3):
                try:
                    resp = openai.chat.completions.create(
                        model='gpt-4o-mini',
                        messages=[{'role': 'user', 'content': prompt}],
                        max_tokens=max_tokens,
                        temperature=temperature,
                    )
                    return resp.choices[0].message.content or ''
                except Exception as e:
                    print(f'Tentativa {attempt+1} falhou: {e}')
                    time.sleep(2 ** attempt)
            return ''
        elif self.backend == 'gpt4all':
            if self.model is None:
                return ''
            try:
                return self.model.generate(prompt,
                                          max_tokens=max_tokens,
                                          temp=temperature)
            except Exception as e:
                print(f'Erro GPT4All: {e}')
                return ''
        else:
            raise ValueError(f'Backend desconhecido: {self.backend}')

# Instanciar — deepseek como padrao
print('Inicializando backends...')
openai_key = os.getenv('OPENAI_API_KEY')
deepseek_key = os.getenv('DEEPSEEK_API_KEY')
llm_deepseek = LLMProvider('deepseek') if deepseek_key else None
llm_gpt4all = LLMProvider('gpt4all')
print(f'DeepSeek: {"OK" if llm_deepseek else "SEM KEY"} | GPT4All: {"OK" if llm_gpt4all else "FALHOU"}')


> **Arquitetura de cada backend:**
>
> - **GPT4All (local):** Modelo Phi-3-mini-4k-instruct.Q4_K_M.gguf (~2 GB) executado
>   localmente na CPU via biblioteca `gpt4all`. Sem internet necessaria. Custos
>   Marginais de eletricidade (~R$ 0,05/hora). Privacidade total — dados nunca
>   saem da máquina.
>
> - **OpenAI GPT-4o-mini (remoto):** API REST acessando servidores OpenAI nos EUA.
>   Resposta em ~500-1500ms (rede). Custo por token. Dados podem ser retidos
>   por 30 dias (politica atual). Superior em qualidade de raciocinio.

## 7.2 Teste de Qualidade (Acurácia + F1)

In [ ]:
TEMPLATE_ZERO = """[PAPEL]
Voce e um farmacologo clinico.

[TAREFA]
Classifique a interacao entre {alvo} e {outro} com base neste contexto:
{contexto}

[FORMATO — OBRIGATORIO]
Responda apenas: {"classe": <0, 1 ou 2>}"""

def parse_response(raw: str) -> int | None:
    if not raw:
        return None
    raw = re.sub(r'```json\s*|\s*```', '', raw).strip()
    try:
        d = json.loads(raw)
        if 'classe' in d:
            return int(d['classe'])
    except:
        m = re.search(r'"classe"\s*:\s*(\d)', raw)
        if m:
            return int(m.group(1))
    return None

# Carregar test set
test_df = pd.read_csv(ANOTACOES_DIR / 'test.csv')
amostra = test_df.head(50)  # 50 pares para teste rapido

def avaliar_backend(llm, nome, max_pares=50):
    print(f'Iniciando avaliacao: {nome} ({max_pares} pares)...')
    if llm is None:
        return None
    preds, labels, json_ok = [], [], 0
    for _, row in amostra.iterrows():
        prompt = TEMPLATE_ZERO.format(
            alvo=row['medicamento_alvo'],
            outro=row['medicamento_outro'],
            contexto=row['contexto'][:200],
        )
        raw = llm.generate(prompt, max_tokens=100)
        parsed = parse_response(raw)
        if parsed is not None:
            preds.append(parsed)
            labels.append(row['classe'])
            json_ok += 1
        if (len(preds)) % 10 == 0:
            print(f'  [{len(preds)}/{max_pares}] {nome}...')
            preds.append(parsed)
            labels.append(row['classe'])
            json_ok += 1
    
    acc = accuracy_score(labels, preds) if preds else 0
    f1_macro = f1_score(labels, preds, average='macro', zero_division=0) if preds else 0
    f1_per = f1_score(labels, preds, average=None, labels=[0,1,2], zero_division=0) if preds else [0,0,0]
    print(f'{nome}: Acc={acc:.2f} | F1={f1_macro:.2f} | JSON={json_ok}/{len(amostra)}')
    return {'backend': nome, 'acc': acc, 'f1_macro': f1_macro,
            'f1_0': f1_per[0], 'f1_1': f1_per[1], 'f1_2': f1_per[2],
            'json_ok': json_ok, 'total': len(amostra)}

result_gpt4all = avaliar_backend(llm_gpt4all, 'GPT4All (Phi-3-mini)')
result_openai = avaliar_backend(llm_openai, 'OpenAI (GPT-4o-mini)')

In [ ]:
# Tabela comparativa de qualidade
resultados = [r for r in [result_gpt4all, result_openai] if r]
if resultados:
    df_qualidade = pd.DataFrame(resultados)
    df_qualidade = df_qualidade[['backend', 'acc', 'f1_macro', 'f1_0', 'f1_1', 'f1_2', 'json_ok', 'total']]
    df_qualidade.columns = ['Backend', 'Acuracia', 'F1 Macro', 'F1 Classe 0', 'F1 Classe 1', 'F1 Classe 2 (GRAVE)', 'JSON Validos', 'Total']
    display(df_qualidade)
    print('\nAnálise: OpenAI é superior em todas as métricas de qualidade.')
    print('GPT4All tem desempenho limitado para textos longos e medicalmente densos.')

## 7.3 Teste de Latência

In [ ]:
CONSULTA_TESTE = "Classifique: Amoxicilina + Ibuprofeno — interação leve ou moderada?"
N_consultas = 20

def medir_latencia(llm, nome):
    print(f'  Latencia: {nome} ({N_consultas} chamadas)...')
    if llm is None:
        return None
    latencias = []
    for _ in range(N_consultas):
        t0 = time.time()
        llm.generate(CONSULTA_TESTE, max_tokens=50)
        latencias.append((time.time() - t0) * 1000)
    return {
        'backend': nome,
        'media': np.mean(latencias),
        'mediana': np.median(latencias),
        'p95': np.percentile(latencias, 95),
        'p99': np.percentile(latencias, 99),
        'min': np.min(latencias),
        'max': np.max(latencias),
    }

lat_gpt4all = medir_latencia(llm_gpt4all, 'GPT4All (CPU)')
lat_openai = medir_latencia(llm_openai, 'OpenAI API')

latencias = [l for l in [lat_gpt4all, lat_openai] if l]
df_lat = pd.DataFrame(latencias)
df_lat = df_lat[['backend', 'media', 'mediana', 'p95', 'p99', 'min', 'max']]
df_lat.columns = ['Backend', 'Media (ms)', 'Mediana (ms)', 'P95 (ms)', 'P99 (ms)', 'Min (ms)', 'Max (ms)']
display(df_lat)

print('\nGPT4All em CPU é mais lento por token gerado, mas sem latência de rede.')
print('OpenAI depende de internet — P99 pode variar bastante.')

In [ ]:
# Grafico de barras — latencia
if latencias:
    fig, ax = plt.subplots(figsize=(8, 5))
    nomes = [l['backend'] for l in latencias]
    medias = [l['media'] for l in latencias]
    p95s = [l['p95'] for l in latencias]
    cores = ['#3498db', '#e74c3c']
    
    x = np.arange(len(nomes))
    ax.bar(x, medias, color=cores, alpha=0.7, label='Media')
    ax.bar(x, p95s, color=cores, alpha=0.4, label='P95')
    ax.set_xticks(x)
    ax.set_xticklabels(nomes)
    ax.set_ylabel('Latência (ms)')
    ax.set_title('Latência: GPT4All vs OpenAI')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 7.4 Análise de Custo

In [ ]:
# Custo GPT4All: eletricidade apenas
# RTX 3050 6GB: ~120W em uso pesado
# R$ 0,70/kWh (anatel 2024)
# 1 hora = 0,12 kWh * 0,70 = R$ 0,084/hora
# Com 500 tokens/consulta: ~0,3s por consulta = 0,000007R$/consulta

custo_gpt4all_por_consulta = 0.000007  # ~R$ 0,00001

# Custo OpenAI GPT-4o-mini (junho 2024)
# $0.15/1M input + $0.60/1M output
input_tokens = 500
output_tokens = 150
custo_input = (input_tokens / 1_000_000) * 0.15  # em dolares
custo_output = (output_tokens / 1_000_000) * 0.60
custo_openai_por_consulta_dolar = custo_input + custo_output

# Conversao dolar→real (R$ 5,00)
taxa = 5.0
custo_openai_por_consulta = custo_openai_por_consulta_dolar * taxa

cenarios = [1000, 10000, 100000]
df_custo = pd.DataFrame({
    'Cenarios': [f'{n:,} consultas/mês' for n in cenarios],
    'GPT4All Local (R$)': [round(n * custo_gpt4all_por_consulta, 2) for n in cenarios],
    'OpenAI API (R$)': [round(n * custo_openai_por_consulta, 2) for n in cenarios],
})
df_custo['Diferença (R$)'] = df_custo['OpenAI API (R$)'] - df_custo['GPT4All Local (R$)']
display(df_custo)

print('\nPara > 10.000 consultas/mês, GPT4All local é mais econômico.')
print('Custo fixo de GPU (RTX 3050 ~R$ 1.500) se paga em ~18 meses.')

## 7.5 Análise de Privacidade

In [ ]:
analise_privacidade = {
    'Dimensao': [
        'Dados saem da máquina',
        'Compatível LGPD',
        'Compatível HIPAA',
        'Dependência de política de terceiros',
        'Retenção pela API',
        'Transferência internacional',
    ],
    'GPT4All Local': [
        'NÃO — 100% local', 'SIM — dado não sai', 'SIM', 'NÃO', 'N/A', 'N/A'
    ],
    'OpenAI API': [
        'SIM — para servidores OpenAI', 'PARCIAL — exige contrato', 'PARCIAL',
        'SIM — política pode mudar', '30 dias (política atual)', 'SIM — exige garantias'
    ],
    'Vencedor': ['Local'] * 6,
}

df_priv = pd.DataFrame(analise_privacidade)
display(df_priv)

print('\n⚠️  Bulário ANVISA é dado PÚBLICO — risco baixo para bulas.')
print('⚠️  Mas a CONSULTA revela: condições de saúde, medicamentos em uso.')
print('🏥 Para hospital/clínica: modelo LOCAL é Obrigatório.')

## 7.6 Análise de Controle e Disponibilidade

In [ ]:
analise_controle = {
    'Dimensao': [
        'Controle de versao do modelo',
        'Funciona offline',
        'Risco de deprecacao',
        'Manutencao necessaria',
        'Disponibilidade (uptime)',
        'Atualizacao automatica',
    ],
    'GPT4All Local': [
        'Total — voce controla', 'SIM — area sem internet', 'Nenhum', 'Atualizacao manual', '100%', 'NAO'],
    'OpenAI API': ['Nenhum — OpenAI decide', 'NAO', 'SIM — GPT-3.5 foi depreciado', 'Zero', '~99.9%', 'SIM — sempre novo'],
    'Vencedor': ['Local', 'Local', 'OpenAI', 'OpenAI', 'OpenAI', 'OpenAI'],
}
df_ctrl = pd.DataFrame(analise_controle)
display(df_ctrl)

## 7.7 Conclusão e Recomendação

In [ ]:
df_conclusao = pd.DataFrame({
    'Dimensao': ['Qualidade (F1)', 'Latência', 'Custo (>10K/mês)', 'Privacidade', 'Controle'],
    'GPT4All Local': ['⭐⭐', '⭐⭐', '⭐⭐⭐', '⭐⭐⭐', '⭐⭐⭐'],
    'OpenAI Remoto': ['⭐⭐⭐', '⭐⭐⭐', '⭐', '⭐', '⭐⭐'],
    'Vencedor': ['OpenAI', 'OpenAI', 'Local', 'Local', 'Local'],
})
display(df_conclusao)

print('\n' + '=' * 60)
print('RECOMENDAÇÃO DO PROJETO')
print('=' * 60)
print('• Notebooks de análise (Fases 5-7): OpenAI — melhor qualidade')
print('• Pipeline RAG em produção: GPT4All local — privacidade + custo')
print('• O codigo de LLMProvider permite trocar de backend em 1 linha')
print('• Para uso hospitalar: modelo local é Obrigatório (LGPD)')